# Notebook setup

In [1]:
from pathlib import Path
import os

# show where the notebook is running
print("CWD before:", Path.cwd())

# Point to your package (adjust if needed)
# e.g. if your modules are under src/, add it to sys.path
import sys
sys.path.append(str(Path.cwd()))  # or Path("src").resolve()

print("CWD after:", Path.cwd())
# from data_processing.logging_utils import logger
# from data_processing.data_setup import create_data_directory

CWD before: /mnt/c/Users/Barrs/OneDrive - mail.tau.ac.il/MLHCproject/test2/mlhc_project/src
CWD after: /mnt/c/Users/Barrs/OneDrive - mail.tau.ac.il/MLHCproject/test2/mlhc_project/src


In [2]:
import pandas as pd
import pickle
from pathlib import Path
from typing import List, Tuple
import numpy as np

# from data_processing.integrated_data_preprocessor import IntegratedICUPreprocessor
# from cohort_data import get_cohort_hadm_ids_and_targets
# from logging_utils import logger
# from data_processing.data_setupd import create_data_directory

# Input CSV file paths
INITIAL_COHORT_CSV = "../csvs/initial_cohort.csv"    # Training/validation patient IDs
TEST_EXAMPLE_CSV = "../csvs/test_example.csv"        # Test set patient IDs

# Output directory for processed data
DATA_DIR = "data"

df_init = pd.read_csv(INITIAL_COHORT_CSV)
df_test = pd.read_csv(TEST_EXAMPLE_CSV)
display(df_init.head()); display(df_test.head())
print("init shape:", df_init.shape, "test shape:", df_test.shape)

# For faster debug runs, sample a small subset (e.g., 100 patients)
INIT_SAMPLE_N = 200
TEST_SAMPLE_N = 50

# init_ids = df_init["subject_id"].astype(int).sample(min(INIT_SAMPLE_N, len(df_init)), random_state=42).tolist()
init_ids = df_init["subject_id"].astype(int).tolist()
test_ids = df_test["subject_id"].astype(int).sample(min(TEST_SAMPLE_N, len(df_test)), random_state=42).tolist()

print(len(init_ids), len(test_ids))

,subject_id
0,22392
1,2847
2,12056
3,25600
4,73125


,subject_id
0,5456
1,1728
2,11199
3,23009
4,15546


init shape: (32513, 1) test shape: (50, 1)
32513 50


## DB access sanity + cohort/targets only

In [3]:
import duckdb
from data_processing.data_extraction import DUCKDB_PATH  # or set your own path here

# db = Path("/mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii.duckdb")
    # assert db.exists(), f"DB not found at {db}"
print("DUCKDB_PATH ->", DUCKDB_PATH)
# con = duckdb.connect(DUCKDB_PATH)
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# con = duckdb.connect(str(db), read_only=True)

# sanity checks
print(con.execute("PRAGMA database_list").fetchdf())
print(con.execute("SHOW TABLES").fetchdf())

DUCKDB_PATH -> /mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii.duckdb
   seq      name                                               file
0  570  mimiciii  /mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii...
                  name
0           ADMISSIONS
1              CALLOUT
2           CAREGIVERS
3          CHARTEVENTS
4            CPTEVENTS
5       DATETIMEEVENTS
6        DIAGNOSES_ICD
7             DRGCODES
8                D_CPT
9      D_ICD_DIAGNOSES
10    D_ICD_PROCEDURES
11             D_ITEMS
12          D_LABITEMS
13            ICUSTAYS
14      INPUTEVENTS_CV
15      INPUTEVENTS_MV
16           LABEVENTS
17  MICROBIOLOGYEVENTS
18          NOTEEVENTS
19        OUTPUTEVENTS
20            PATIENTS
21       PRESCRIPTIONS
22  PROCEDUREEVENTS_MV
23      PROCEDURES_ICD
24            SERVICES
25           TRANSFERS


In [ ]:
from data_processing.cohort_data import COHORT_SQL

# Register subject IDs as temporary table for SQL query
con.register("tmp_subject_ids", pd.DataFrame({"subject_id": init_ids}))

# Execute cohort SQL to get filtered admissions and target labels
df = con.execute(COHORT_SQL).fetchdf()
print(df.head())
print(df.shape)

# Extract admission IDs and target matrix
hadm_ids = df["hadm_id"].tolist()
targets = df[["mortality_event", "los_event", "readmission_event"]].reset_index(drop=True).values

   hadm_id  mortality_event  los_event  readmission_event
0   100003                0          0                  0
1   100006                0          1                  0
2   100007                0          1                  0
3   100009                0          0                  0
4   100010                0          0                  0
(22489, 4)


In [6]:
# Build a base table with everything we need
# returns all intermediate columns you need to check rules yourself (in Python):
#  subject_id, hadm_id, admittime, dischtime, deathtime, age, los_hours, has_chartevents_data, admission_rank, discharge_to_death_hours, discharge_to_readmission_hours, plus a computed helper died_within_54h.
#  It does not filter the cohort and does not output targets
base_sql = r"""
WITH ordered AS (
  SELECT
      a.subject_id::INTEGER            AS subject_id,
      a.hadm_id::INTEGER               AS hadm_id,
      a.admittime::TIMESTAMP           AS admittime,
      a.dischtime::TIMESTAMP           AS dischtime,
      a.deathtime::TIMESTAMP           AS deathtime,
      a.has_chartevents_data::INTEGER  AS has_chartevents_data,
      EXTRACT(year FROM AGE(a.admittime::TIMESTAMP, p.dob::TIMESTAMP))::INTEGER AS age,
      -- hospital LOS in hours
      EXTRACT(epoch FROM (a.dischtime::TIMESTAMP - a.admittime::TIMESTAMP)) / 3600.0 AS los_hours,
      -- death within first 54h of *admission* (Rule 5 exclusion)
      CASE
        WHEN a.deathtime IS NOT NULL
             AND EXTRACT(epoch FROM (a.deathtime::TIMESTAMP - a.admittime::TIMESTAMP)) / 3600.0 <= 54
        THEN 1 ELSE 0
      END AS died_within_54h,
      -- time to death after *discharge* (mortality target window)
      EXTRACT(epoch FROM (p.dod::TIMESTAMP - a.dischtime::TIMESTAMP)) / 3600.0 AS discharge_to_death_hours,
      -- time to next admission
      EXTRACT(epoch FROM (LEAD(a.admittime::TIMESTAMP) OVER (PARTITION BY a.subject_id ORDER BY a.admittime)
                          - a.dischtime::TIMESTAMP)) / 3600.0 AS discharge_to_readmission_hours,
      ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime) AS admission_rank
  FROM admissions a
  JOIN patients  p ON a.subject_id = p.subject_id
  WHERE a.subject_id::INTEGER IN (SELECT subject_id FROM tmp_subject_ids)
)
SELECT *
FROM ordered
ORDER BY subject_id, admittime
"""
base_df = con.execute(base_sql).fetchdf()
base_df.head(20)
# print(base_df.shape)

,subject_id,hadm_id,admittime,dischtime,deathtime,has_chartevents_data,age,los_hours,died_within_54h,discharge_to_death_hours,discharge_to_readmission_hours,admission_rank
0,2,163353,2138-07-17 19:04:00,2138-07-21 15:48:00,NaT,1,0,92.733333,0,NaN,NaN,1
1,3,145834,2101-10-20 19:08:00,2101-10-31 13:58:00,NaT,1,76,258.833333,0,5410.033333,NaN,1
2,4,185777,2191-03-16 00:28:00,2191-03-23 18:41:00,NaT,1,47,186.216667,0,NaN,NaN,1
3,5,178980,2103-02-02 04:31:00,2103-02-04 12:15:00,NaT,1,0,55.733333,0,NaN,NaN,1
4,7,118037,2121-05-23 15:05:00,2121-05-27 11:57:00,NaT,1,0,92.866667,0,NaN,NaN,1
5,8,159514,2117-11-20 10:22:00,2117-11-24 14:20:00,NaT,1,0,99.966667,0,NaN,NaN,1
6,9,150750,2149-11-09 13:06:00,2149-11-14 10:15:00,2149-11-14 10:15:00,1,41,117.150000,0,-10.250000,NaN,1
7,11,194540,2178-04-16 06:18:00,2178-05-11 19:00:00,NaT,1,50,612.700000,0,4469.000000,NaN,1
8,12,112213,2104-08-07 10:15:00,2104-08-20 02:57:00,2104-08-20 02:57:00,1,72,304.700000,0,-2.950000,NaN,1
9,16,103251,2178-02-03 06:35:00,2178-02-05 10:51:00,NaT,1,0,52.266667,0,NaN,NaN,1


In [7]:
# Recreate the 5 rules in Python (ground truth inclusion)
MIN_AGE, MAX_AGE = 18, 89
MIN_LOS_HOURS = 54
MORTALITY_EVENT_HOURS = 720
LOS_EVENT_HOURS = 168
READMISSION_EVENT_HOURS = 720

def apply_rules(df):
    m1 = df["admission_rank"] == 1
    m2 = df["age"].between(MIN_AGE, MAX_AGE, inclusive="both")
    m3 = df["los_hours"] >= MIN_LOS_HOURS
    m4 = df["has_chartevents_data"] == 1
    m5 = df["died_within_54h"] == 0

    keep_mask = m1 & m2 & m3 & m4 & m5
    sizes = {
        "start": len(df),
        "rule1_first": int(m1.sum()),
        "rule2_age": int((m1 & m2).sum()),
        "rule3_los": int((m1 & m2 & m3).sum()),
        "rule4_chartevents": int((m1 & m2 & m3 & m4).sum()),
        "rule5_no_early_death": int(keep_mask.sum()),
    }
    # print(df[m3 & ~m5])
    return df[keep_mask].copy(), sizes

# print(base_df[base_df["died_within_54h"] == 1])
# print(base_df[base_df[base_df["los_hours"] >= MIN_LOS_HOURS]])
expected_cohort_df, sizes = apply_rules(base_df)
sizes, expected_cohort_df.shape


({'start': 41244,
  'rule1_first': 32513,
  'rule2_age': 25548,
  'rule3_los': 22927,
  'rule4_chartevents': 22493,
  'rule5_no_early_death': 22489},
 (22489, 12))

In [8]:
# Compute ground truth targets in Python
gt = expected_cohort_df.copy()

# Mortality: death within 30 days *after discharge* (using patients.dod vs dischtime)
gt["mortality_event"] = (gt["discharge_to_death_hours"] <= MORTALITY_EVENT_HOURS).astype(int)

# LOS>7 days target: NOTE this uses *hospital* LOS. If you intended ICU LOS, change source.
gt["los_event"] = (gt["los_hours"] > LOS_EVENT_HOURS).astype(int)

# Readmission within 30 days after discharge
gt["readmission_event"] = (gt["discharge_to_readmission_hours"] <= READMISSION_EVENT_HOURS).astype(int)

gt_targets = gt[["hadm_id","mortality_event","los_event","readmission_event"]].sort_values("hadm_id").reset_index(drop=True)
gt_targets.head()


,hadm_id,mortality_event,los_event,readmission_event
0,100003,0,0,0
1,100006,0,1,0
2,100007,0,1,0
3,100009,0,0,0
4,100010,0,0,0


In [11]:
from data_processing.cohort_data import get_cohort_hadm_ids_and_targets, COHORT_SQL
# Run COHORT_SQL and compare
sql_df = con.execute(COHORT_SQL).fetchdf()
sql_targets = sql_df[["hadm_id","mortality_event","los_event","readmission_event"]].sort_values("hadm_id").reset_index(drop=True)

# 1) Inclusion check: which HADM_IDs the SQL kept vs. Python rules
gt_hadm = set(gt_targets["hadm_id"].astype(int))
sql_hadm = set(sql_targets["hadm_id"].astype(int))

extra = sorted(sql_hadm - gt_hadm)    # in SQL but should be excluded
missing = sorted(gt_hadm - sql_hadm)  # expected by rules but not in SQL

print(f"GT size={len(gt_hadm)}  |  SQL size={len(sql_hadm)}")
print(f"Extra: {len(extra)}  Missing: {len(missing)}")
if extra:  display(base_df[base_df.hadm_id.isin(extra)].head(10))
if missing: display(base_df[base_df.hadm_id.isin(missing)].head(10))

# 2) Target equality for the intersection
common = sorted(gt_hadm & sql_hadm)
gt_common = gt_targets[gt_targets.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)
sql_common = sql_targets[sql_targets.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)

mismatch = (gt_common[["mortality_event","los_event","readmission_event"]].values !=
            sql_common[["mortality_event","los_event","readmission_event"]].values)
n_mismatch = int(mismatch.any(axis=1).sum())

print(f"Target mismatches on common HADM_IDs: {n_mismatch} / {len(common)}")
if n_mismatch:
    bad = gt_common.loc[mismatch.any(axis=1)]
    bad = bad.merge(sql_common, on="hadm_id", suffixes=("_gt","_sql"))
    display(bad.head(20))


GT size=22489  |  SQL size=22489
Extra: 0  Missing: 0
Target mismatches on common HADM_IDs: 0 / 22489


In [14]:
# 1) Get the ORIGINAL outputs (what your pipeline returns today)
hadm_ids_orig, targets_orig = get_cohort_hadm_ids_and_targets(con, init_ids)
print("cohort #hadm:", len(hadm_ids_orig))
print("targets shape:", targets_orig.shape)
pd.DataFrame(targets_orig, columns=["mortality","los>7d","readm≤30d"]).head()

# 2) Convert them to a tidy DataFrame (so we can compare apples-to-apples)
orig_df = (
    pd.DataFrame({
        "hadm_id": hadm_ids_orig,
        "mortality_event": targets_orig[:, 0].astype(int),
        "los_event": targets_orig[:, 1].astype(int),
        "readmission_event": targets_orig[:, 2].astype(int),
    })
    # .sort_values("hadm_id")
    .reset_index(drop=True)
)

# 4) Cohort inclusion diffs (who is in vs. who should be in)
orig_set = set(orig_df["hadm_id"].astype(int))
gt_set   = set(gt_targets["hadm_id"].astype(int))

extra   = sorted(orig_set - gt_set)   # present in ORIGINAL but not in GT (over-inclusion)
missing = sorted(gt_set - orig_set)   # present in GT but not in ORIGINAL (over-filtering)

print(f"Original size = {len(orig_set)} | GT size = {len(gt_set)}")
print(f"Extra (in ORIGINAL, not GT): {len(extra)}")
print(f"Missing (in GT, not ORIGINAL): {len(missing)}")

if extra:
    display(base_df[base_df.hadm_id.isin(extra)][
        ["subject_id","hadm_id","admittime","dischtime","deathtime","age","los_hours","has_chartevents_data","admission_rank"]
    ].head(10))

if missing:
    display(base_df[base_df.hadm_id.isin(missing)][
        ["subject_id","hadm_id","admittime","dischtime","deathtime","age","los_hours","has_chartevents_data","admission_rank"]
    ].head(10))

# 5) Target label comparison on the intersection
common = sorted(orig_set & gt_set)
orig_common = orig_df[orig_df.hadm_id.isin(common)].reset_index(drop=True)
gt_common   = gt_targets[gt_targets.hadm_id.isin(common)].reset_index(drop=True)

# Vectorized mismatch check
cols = ["mortality_event","los_event","readmission_event"]
mismatch_mask = (orig_common[cols].values != gt_common[cols].values)
rows_with_any_mismatch = mismatch_mask.any(axis=1)
n_rows_bad = int(rows_with_any_mismatch.sum())

print(f"Target mismatches on common HADM_IDs: {n_rows_bad} / {len(common)}")
if n_rows_bad:
    bad = (
        orig_common.loc[rows_with_any_mismatch, ["hadm_id"] + cols]
        .merge(gt_common.loc[rows_with_any_mismatch, ["hadm_id"] + cols],
               on="hadm_id", suffixes=("_orig", "_gt"))
    )
    display(bad.head(20))

15:29:14.886 Started get_cohort_hadm_ids_and_targets
15:29:14.968 Finished get_cohort_hadm_ids_and_targets
cohort #hadm: 22489
targets shape: (22489, 3)
Original size = 22489 | GT size = 22489
Extra (in ORIGINAL, not GT): 0
Missing (in GT, not ORIGINAL): 0
Target mismatches on common HADM_IDs: 0 / 22489


In [8]:
# label distribution check (stratification sanity)
lab = pd.DataFrame(targets, columns=["mortality","los","readm"])
lab["combo"] = (lab["mortality"].astype(int).astype(str) +
                lab["los"].astype(int).astype(str) +
                lab["readm"].astype(int).astype(str))
lab["combo"].value_counts(normalize=True).rename("freq").to_frame()

,freq
combo,
010,0.524590
000,0.327869
110,0.065574
100,0.049180
001,0.032787


## Static features only

In [9]:
from data_processing.static_data import get_static_data, STATIC_COLUMNS

static_raw = get_static_data(con, hadm_ids)
print("static_raw shape:", static_raw.shape)

# peek as DataFrame with original column ordering
# from yourpkg.static_data import STATIC_COLUMNS
pd.DataFrame(static_raw, columns=STATIC_COLUMNS).head()

16:10:14.973 Started get_static_data
16:10:59.181 Finished get_static_data
static_raw shape: (61, 17)


,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,height,weight,received_vasopressor,recieved_mechanical_ventilation,received_rrt,received_sedation,received_antibiotic,reached_icu
0,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Government,RUSS,OTHER,DIVORCED,OTHER,0,72,NaN,NaN,0,1,0,1,1,1
1,EMERGENCY,EMERGENCY ROOM ADMIT,Medicare,PTUN,UNOBTAINABLE,MARRIED,UNKNOWN/NOT SPECIFIED,1,80,173.0,88.495871,1,1,1,0,1,1
2,EMERGENCY,CLINIC REFERRAL/PREMATURE,Private,missing,NOT SPECIFIED,MARRIED,WHITE,1,65,NaN,NaN,0,1,1,0,0,1
3,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,CATHOLIC,MARRIED,WHITE,0,78,NaN,79.1,1,1,0,1,1,0
4,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,CATHOLIC,MARRIED,WHITE,0,61,NaN,NaN,0,1,1,1,1,1


## Time-series features only

In [ ]:
from data_processing.timeseries_data import get_timeseries_data, TIMESERIES_COLUMNS

timeseries_data, timeseries_missingness = get_timeseries_data(con, hadm_ids)
# con.close()

print("ts_data:", timeseries_data.shape, "ts_miss:", timeseries_missingness.shape)
# tiny peek
N, H, F = timeseries_data.shape
timeseries_data[0, :3, :8]  # first patient, first 3 hours, first 8 features
pd.DataFrame(timeseries_data, columns=TIMESERIES_COLUMNS).head()

16:12:31.771 Started get_timeseries_data
16:12:40.256 Finished get_timeseries_data
ts_data: (61, 48, 216) ts_miss: (61, 48, 216)


ValueError: Must pass 2-d input. shape=(61, 48, 216)